In [39]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
stats = pd.read_csv("../data/season_stats.csv")
stats.head()

,Unnamed: 0,week,name,opp_name,tm_score,opp_score,rush_yds,pass_yds
0,12,18,Cowboys,Football Team,38,10,131.0,309
1,13,18,Rams,49ers,21,20,109.0,163
2,14,18,Chiefs,Chargers,13,12,123.0,154
3,15,18,Raiders,Broncos,27,14,129.0,244
4,16,18,Titans,Jaguars,28,20,175.0,168


there are null values in the rush_yds column all for week 12

In [3]:
stats2 = stats.copy()
stats3 = stats.copy()
stats4 = stats.copy()

a simple solution  
replacing null values with the average non null values in the rush_yds column  
Accounts for team identity but ignores opponent or game context.  

Pros:
Simple and fast — one line of code  
No risk of data leakage or overfitting  
Works even if a team has very few games in the dataset  

Cons:  
Ignores team identity — a power-run team like the Titans gets the same fill value as a pass-heavy team like the Chiefs  
All 16 nulls get the exact same value, which adds no variation and can distort distributions or model training  
The mean is sensitive to outliers (e.g., the Dolphins' 350-yard game in week 3 pulls it up)


In [4]:
mean_rush_yds = stats2['rush_yds'].mean()
stats2['rush_yds'] = stats2['rush_yds'].fillna(round(mean_rush_yds)).astype(int)
stats2['rush_yds'].isnull().sum()

0

In [5]:
# verifying datatype
stats2.dtypes

Unnamed: 0     int64
week           int64
name          object
opp_name      object
tm_score       int64
opp_score      int64
rush_yds       int32
pass_yds       int64
dtype: object

In [6]:
# verifying nulls were filled and with the same value
stats2[stats2['week'] == 12]

,Unnamed: 0,week,name,opp_name,tm_score,opp_score,rush_yds,pass_yds
92,104,12,Bears,Vikings,12,10,127,217
93,105,12,Ravens,Chargers,20,10,127,177
94,106,12,Chiefs,Raiders,31,17,127,298
95,107,12,Eagles,Bills,37,34,127,200
96,108,12,Titans,Panthers,17,10,127,185
97,109,12,Giants,Patriots,10,7,127,191
98,110,12,Jaguars,Texans,24,21,127,364
99,111,12,Broncos,Browns,29,12,127,134
100,112,12,Rams,Cardinals,37,14,127,229
101,113,12,Colts,Buccaneers,27,20,127,251


In [7]:
# verifying nulls were filled
print(stats.isnull().sum())
print(stats2.isnull().sum())

Unnamed: 0     0
week           0
name           0
opp_name       0
tm_score       0
opp_score      0
rush_yds      16
pass_yds       0
dtype: int64
Unnamed: 0    0
week          0
name          0
opp_name      0
tm_score      0
opp_score     0
rush_yds      0
pass_yds      0
dtype: int64


Slightly more complex  
replacing null values with the rysh_yrds team average  

Pros:  
More personalized — accounts for each team's actual rushing tendencies  
Simple to implement and still easy to interpret  
Better than global mean when teams vary significantly in rushing style 

Cons:  
Still ignores game context — doesn't account for opponent strength, score, or week  
Teams with fewer games in the dataset have less reliable averages  
All nulls for the same team get the same fill value, still adding no variation within that team  


In [8]:
stats3['rush_yds'] = stats3.groupby('name')['rush_yds'].transform(lambda x: x.fillna(round(x.mean()))).astype(int)
stats3['rush_yds'].isnull().sum()

0

In [9]:
stats3.dtypes

Unnamed: 0     int64
week           int64
name          object
opp_name      object
tm_score       int64
opp_score      int64
rush_yds       int32
pass_yds       int64
dtype: object

In [10]:
stats3[stats3['week'] == 12]

,Unnamed: 0,week,name,opp_name,tm_score,opp_score,rush_yds,pass_yds
92,104,12,Bears,Vikings,12,10,178,217
93,105,12,Ravens,Chargers,20,10,162,177
94,106,12,Chiefs,Raiders,31,17,108,298
95,107,12,Eagles,Bills,37,34,137,200
96,108,12,Titans,Panthers,17,10,144,185
97,109,12,Giants,Patriots,10,7,128,191
98,110,12,Jaguars,Texans,24,21,124,364
99,111,12,Broncos,Browns,29,12,110,134
100,112,12,Rams,Cardinals,37,14,131,229
101,113,12,Colts,Buccaneers,27,20,121,251


In [18]:
# verifing two teams to see if the rush_yds were filled in correctly

# stats3[stats3['name'] == 'Bears']
stats3[stats3['name'] == 'Ravens']


,Unnamed: 0,week,name,opp_name,tm_score,opp_score,rush_yds,pass_yds
19,31,17,Ravens,Dolphins,56,19,160,340
32,44,16,Ravens,49ers,33,19,102,252
54,66,15,Ravens,Jaguars,23,7,251,171
68,80,14,Ravens,Rams,37,31,139,316
93,105,12,Ravens,Chargers,20,10,162,177
121,133,11,Ravens,Bengals,34,20,157,264
137,149,9,Ravens,Seahawks,37,3,298,225
162,174,8,Ravens,Cardinals,31,24,130,157
169,181,7,Ravens,Lions,38,6,146,357
183,195,6,Ravens,Titans,24,16,139,223


most complex  
replacing null rush_yds with values predicted by using the values in the name, tm_score, opp_score, pass_yds columns  

Pros:  
Uses actual game context (score, passing yards, team identity) to make each prediction unique  
More realistic than mean imputation — a high-scoring game with low pass yards will predict differently than a low-scoring game  
Generalizes reasonably well with enough training data  

Cons:  
Assumes a linear relationship between features and rush_yds — that may not hold (e.g., rushing and passing yards can be inversely related in non-linear ways)  
Trained on 17 other weeks, predicting week 12 specifically — if week 12 had unusual conditions (weather, injuries), the model won't know  
One-hot encoding name adds 32 columns but each team only has ~17 rows of training data, which is thin  
Harder to explain to a non-technical audience than a simple average  

In [12]:
features = ['name', 'tm_score', 'opp_score', 'pass_yds']

# converting the team names to dummy variables and keeping the other features as is
stats4_encoded = pd.get_dummies(stats4[features])

# spliting into known and unknown rows, making a t/f column, rows with a value with go to training, the nulls go to predict
known = stats4['rush_yds'].notna()
X_train = stats4_encoded[known]
y_train = stats4['rush_yds'][known]
X_predict = stats4_encoded[~known]

# training the model
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_predict)

# predict and fill the nulls with the predicted values
stats4.loc[~known, 'rush_yds'] = predictions.round()
stats4['rush_yds'] = stats4['rush_yds'].astype(int)
stats4['rush_yds'].isnull().sum()

0

In [13]:
stats4.dtypes

Unnamed: 0     int64
week           int64
name          object
opp_name      object
tm_score       int64
opp_score      int64
rush_yds       int32
pass_yds       int64
dtype: object

In [14]:
stats4[stats4['week'] == 12]

,Unnamed: 0,week,name,opp_name,tm_score,opp_score,rush_yds,pass_yds
92,104,12,Bears,Vikings,12,10,138,217
93,105,12,Ravens,Chargers,20,10,150,177
94,106,12,Chiefs,Raiders,31,17,116,298
95,107,12,Eagles,Bills,37,34,165,200
96,108,12,Titans,Panthers,17,10,137,185
97,109,12,Giants,Patriots,10,7,114,191
98,110,12,Jaguars,Texans,24,21,82,364
99,111,12,Broncos,Browns,29,12,142,134
100,112,12,Rams,Cardinals,37,14,164,229
101,113,12,Colts,Buccaneers,27,20,118,251


In [19]:
# stats4[stats4['name'] == 'Bears']
stats4[stats4['name'] == 'Ravens']

,Unnamed: 0,week,name,opp_name,tm_score,opp_score,rush_yds,pass_yds
19,31,17,Ravens,Dolphins,56,19,160,340
32,44,16,Ravens,49ers,33,19,102,252
54,66,15,Ravens,Jaguars,23,7,251,171
68,80,14,Ravens,Rams,37,31,139,316
93,105,12,Ravens,Chargers,20,10,150,177
121,133,11,Ravens,Bengals,34,20,157,264
137,149,9,Ravens,Seahawks,37,3,298,225
162,174,8,Ravens,Cardinals,31,24,130,157
169,181,7,Ravens,Lions,38,6,146,357
183,195,6,Ravens,Titans,24,16,139,223


In [21]:
# wanted to see the difference in the two approaches value
week12_stats3 = stats3[stats3['week'] == 12][['name', 'rush_yds']].rename(columns={'rush_yds': 'rush_yds_team_avg'})
week12_stats4 = stats4[stats4['week'] == 12][['name', 'rush_yds']].rename(columns={'rush_yds': 'rush_yds_regression'})

comparison = week12_stats3.merge(week12_stats4, on='name')
comparison['difference'] = comparison['rush_yds_regression'] - comparison['rush_yds_team_avg']
comparison.sort_values('difference', ascending=False)

,name,rush_yds_team_avg,rush_yds_regression,difference
8,Rams,131,164,33
7,Broncos,110,142,32
3,Eagles,137,165,28
11,Falcons,143,161,18
12,Dolphins,142,157,15
13,49ers,153,165,12
15,Cowboys,120,132,12
2,Chiefs,108,116,8
14,Packers,133,138,5
9,Colts,121,118,-3


Testing  
Mean absolute error - The average absolute error among predictions  
Mean squared error - The error of each prediction squared (Punishes larger
errors more)

In [22]:
actual = pd.read_csv("../data/actual.csv")
actual.head()

,week,name,actual_rush_yds
0,18,Cowboys,131
1,18,Rams,109
2,18,Chiefs,123
3,18,Raiders,129
4,18,Titans,175


In [ ]:
# using chat_gpt

# found missing values in the rush_yds column, so we will only evaluate the rows where rush_yds is missing
missing_mask = stats['rush_yds'].isna()

# 
mae_stats2 = mean_absolute_error(
    actual.loc[missing_mask, 'actual_rush_yds'],
    stats2.loc[missing_mask, 'rush_yds']
)

mae_stats3 = mean_absolute_error(
    actual.loc[missing_mask, 'actual_rush_yds'],
    stats3.loc[missing_mask, 'rush_yds']
)

mae_stats4 = mean_absolute_error(
    actual.loc[missing_mask, 'actual_rush_yds'],
    stats4.loc[missing_mask, 'rush_yds']
)

print(f"Stats2 MAE: {mae_stats2:.2f}")
print(f"Stats3 MAE: {mae_stats3:.2f}")
print(f"Stats4 MAE: {mae_stats4:.2f}")

Stats2 MAE: 48.88
Stats3 MAE: 46.38
Stats4 MAE: 35.81


Using mean absolute error the best prediction was using regression  mae 35.81  

Using the average by did only preformed slightly better mae 46.38 than using the average for all teams to fill in the missing values mae 48.88

In [40]:
# using claude

missing_mask = stats['rush_yds'].isna()

mse_stats2 = mean_squared_error(actual.loc[missing_mask, 'actual_rush_yds'], stats2.loc[missing_mask, 'rush_yds'])
mse_stats3 = mean_squared_error(actual.loc[missing_mask, 'actual_rush_yds'], stats3.loc[missing_mask, 'rush_yds'])
mse_stats4 = mean_squared_error(actual.loc[missing_mask, 'actual_rush_yds'], stats4.loc[missing_mask, 'rush_yds'])

print(f"Stats2 MSE (global mean):  {mse_stats2:.2f}")
print(f"Stats3 MSE (team avg):     {mse_stats3:.2f}")
print(f"Stats4 MSE (regression):   {mse_stats4:.2f}")

Stats2 MSE (global mean):  3052.38
Stats3 MSE (team avg):     2697.00
Stats4 MSE (regression):   1705.81


Using mean squared error regression also preformed the best mse 1705.81  
Team average mse was 2697.00, and the global mean preformed the worse at mse 3052.38

Do you think the more complex null replacement methods always/usually/sometimes/rarely perform better than the less complex ones?  

>I think it depends on the context. In this example the column was important, but sometimes the column could be null and be non material, and a simpler replacement could be used. Also I think the amount of data you have to determine the replacement method, and the feature relationships makes a difference. 

In the most complex method we chose some columns to include and some to exclude.  Why did we do this?    

>To get feature data to use in the prediction method. The columns used were related to rushing yards. For example, week was excluded because it was not deemed to be relevant to determine how many rushing yards there were in the game.  Too many features can add noise to the model making it less useful for prediction. 

Would Claude be good at differentiating which columns to include?

>I asked claude what it would have used it said it would have used the same columns, team because it is the id, pass_yds due to the inverse relationship between rush_yds, and the scores. I think if claude had the data and use case it would give a good base line of features to include and then it might have to be tested and tuned after. So yes I think it would be a good starting point to get base features. (chat gpt i would feel less confident with just because of the back and forth i had with it for the mae)

Would understanding relationships among columns help in the process of choosing which to include?  

>I think it would. The stronger relationship the columns have with the value being predicted should help give a better predicted value.  
  
Do you think there is ever a down-side to including too many columns or do you think more data should mean more accuracy?    

>Yes. Including too much can create noise in the model, and lead to less accurate results. 




If we didn't have the actual values to test our results which ways could we
use Claude to test the accuracy?  
>I think that you could use rows that you did have a value and pretend you didnt, and see how the results would be compared to the actual values that were in the original dataset. Could also look at box plot i think it is called to see if the predicted values fall in normal range and not be an outlier for the team. 